In [0]:
# Importamos las librerias necesarias
from pyspark.sql.functions import *

from datetime import datetime, timedelta
from pyspark.sql.types import *

import random

In [0]:
def generate_customer_spark(cantidad):
    """
    Genera datos sintéticos de clientes usando operaciones nativas de Spark.
    Mucho más rápido que loops de Python para grandes volúmenes.
    """
    
    # Generamos el DataFrame base con IDs usando spark.range (operación distribuida)
    df_base = spark.range(1, cantidad + 1).withColumnRenamed("id", "customer_id")
    
    # Definimos arrays de valores para selección aleatoria
    first_names = ["Juan", "Manuel", "Maria", "Pedro", "Jose", "Ana", "Carlos", "Laura", "Sofia", "Diego", 
                   "Luis", "Fernando", "Andres", "Camila", "Valentina", "Santiago", "Daniela", "Gabriela", 
                   "Javier", "Natalia", "Sebastian", "Valeria", "Isabella", "Alejandro", "Mariana", "Miguel", 
                   "Paula", "Ricardo", "Carolina", "Felipe"]
    
    last_names = ["Rodriguez", "Gonzalez", "Perez", "Martinez", "Garcia", "Lopez", "Sanchez", "Jimenez", 
                  "Gomez", "Hernandez", "Torres", "Diaz", "Vargas", "Castro", "Rojas", "Fernandez", 
                  "Morales", "Ramirez", "Silva", "Mendoza", "Ortiz", "Ruiz"]
    
    cities = ["Bogota", "Medellin", "Cali", "Barranquilla", "Cartagena", "Bucaramanga", 
              "Cucuta", "Santa Marta", "Pereira", "Manizales", "Ibague", "Pasto"]
    
    professions = ["Programador", "Analista", "Ingeniero", "Contador", "Abogado", "Medico", 
                   "Enfermero", "Psicologo", "Arquitecto", "Diseñador", "Vendedor", "Gerente", 
                   "Consultor", "Profesor", "Data Engineer", "Auditor", "Marketing Manager"]
    
    education_levels = ["Bachillerato", "Tecnico", "Tecnologia", "Universitario", "Especializacion", "Maestria", "Doctorado"]
    
    company_sectors = ["Tecnologia", "Salud", "Educacion", "Financiero", "Retail", "Manufactura", 
                       "Construccion", "Servicios", "Gobierno", "Telecomunicaciones", "Agricultura", "Mineria"]
    
    payment_methods = ["Efectivo", "Tarjeta Credito", "Tarjeta Debito", "Transferencia", "PSE", "Nequi", "Daviplata"]
    
    contract_types = ["Indefinido", "Fijo", "Obra o Labor", "Prestacion de Servicios", "Freelance"]
    
    marital_status_list = ["Soltero", "Casado", "Divorciado", "Viudo", "Union Libre"]
    
    risk_categories = ["Bajo", "Medio", "Alto", "Muy Alto"]
    
    # Usamos funciones de Spark SQL para generar datos aleatorios de forma distribuida
    df = df_base \
        .withColumn("first_name", expr(f"element_at(array{tuple(first_names)}, int(rand() * {len(first_names)}) + 1)")) \
        .withColumn("last_name", expr(f"element_at(array{tuple(last_names)}, int(rand() * {len(last_names)}) + 1)")) \
        .withColumn("age", expr("int(rand() * 47) + 18")) \
        .withColumn("gender", expr("case when rand() > 0.5 then 'M' else 'F' end")) \
        .withColumn("city", expr(f"element_at(array{tuple(cities)}, int(rand() * {len(cities)}) + 1)")) \
        .withColumn("profession", expr(f"element_at(array{tuple(professions)}, int(rand() * {len(professions)}) + 1)")) \
        .withColumn("education_level", expr(f"element_at(array{tuple(education_levels)}, int(rand() * {len(education_levels)}) + 1)")) \
        .withColumn("contract_type", expr(f"element_at(array{tuple(contract_types)}, int(rand() * {len(contract_types)}) + 1)")) \
        .withColumn("company_sector", expr(f"element_at(array{tuple(company_sectors)}, int(rand() * {len(company_sectors)}) + 1)")) \
        .withColumn("payment_method", expr(f"element_at(array{tuple(payment_methods)}, int(rand() * {len(payment_methods)}) + 1)")) \
        .withColumn("marital_status", expr(f"element_at(array{tuple(marital_status_list)}, int(rand() * {len(marital_status_list)}) + 1)")) \
        .withColumn("dependents_count", expr("int(rand() * 5)")) \
        .withColumn("has_vehicle", expr("rand() > 0.4")) \
        .withColumn("has_property", expr("rand() > 0.6")) \
        .withColumn("credit_score", expr("int(rand() * 550) + 300")) \
        .withColumn("risk_category", expr(f"element_at(array{tuple(risk_categories)}, int(rand() * {len(risk_categories)}) + 1)")) \
        .withColumn("is_active", expr("rand() > 0.3")) \
        .withColumn("registration_date", expr("date_add('2020-01-01', int(rand() * 1460))")) \
        .withColumn("years_employed", expr("int(rand() * 20) + 1")) \
        .withColumn("company_size", expr("case when rand() < 0.3 then 'Pequeña' when rand() < 0.7 then 'Mediana' else 'Grande' end")) \
        .withColumn("remote_work", expr("rand() > 0.5"))
    
    # Agregamos mapeo de departamento usando join (más eficiente que UDF)
    dept_data = [
        ("Bogota", "Cundinamarca"), ("Medellin", "Antioquia"), ("Cali", "Valle del Cauca"),
        ("Barranquilla", "Atlantico"), ("Cartagena", "Bolivar"), ("Bucaramanga", "Santander"),
        ("Cucuta", "Norte de Santander"), ("Santa Marta", "Magdalena"), ("Pereira", "Risaralda"),
        ("Manizales", "Caldas"), ("Ibague", "Tolima"), ("Pasto", "Narino")
    ]
    df_dept = spark.createDataFrame(dept_data, ["city", "department"])
    df = df.join(df_dept, "city", "left")
    
    # Calculamos campos correlacionados usando lógica de Spark SQL
    df = df \
        .withColumn("education_multiplier", 
                    expr("""case 
                        when education_level = 'Bachillerato' then 0.8
                        when education_level = 'Tecnico' then 0.9
                        when education_level = 'Tecnologia' then 1.0
                        when education_level = 'Universitario' then 1.3
                        when education_level = 'Especializacion' then 1.6
                        when education_level = 'Maestria' then 2.0
                        when education_level = 'Doctorado' then 2.5
                        else 1.0 end""")) \
        .withColumn("base_salary", expr("int(rand() * 60000) + 40000")) \
        .withColumn("value_day", expr("int(base_salary * education_multiplier)")) \
        .withColumn("days_worked", expr("int(rand() * 30) + 1")) \
        .withColumn("monthly_income", expr("value_day * days_worked")) \
        .withColumn("bonus_percentage", expr("round(rand() * 0.20, 2)")) \
        .withColumn("annual_bonus", expr("int(monthly_income * 12 * bonus_percentage)")) \
        .withColumn("tax_percentage", expr("case when monthly_income < 2000000 then 0.0 when monthly_income < 5000000 then 0.19 else 0.33 end")) \
        .withColumn("monthly_tax", expr("int(monthly_income * tax_percentage)")) \
        .withColumn("net_monthly_income", expr("monthly_income - monthly_tax")) \
        .withColumn("transactions_count", expr("case when is_active then int(rand() * 50) else int(rand() * 5) end")) \
        .withColumn("avg_transaction_value", expr("case when transactions_count > 0 then int(rand() * 450000) + 50000 else 0 end")) \
        .withColumn("total_spent", expr("transactions_count * avg_transaction_value")) \
        .withColumn("savings_rate", expr("round(rand() * 0.30, 2)")) \
        .withColumn("monthly_savings", expr("int(net_monthly_income * savings_rate)")) \
        .withColumn("debt_amount", expr("int(rand() * 50000000)")) \
        .withColumn("debt_to_income_ratio", expr("round(debt_amount / (monthly_income * 12), 2)")) \
        .withColumn("last_purchase_date", expr("case when is_active then date_add(registration_date, int(rand() * 365)) else null end")) \
        .withColumn("last_login_date", expr("case when is_active then date_add(last_purchase_date, int(rand() * 30)) else null end")) \
        .withColumn("days_since_last_purchase", expr("datediff(current_date(), last_purchase_date)")) \
        .withColumn("customer_tenure_days", expr("datediff(current_date(), registration_date)")) \
        .withColumn("purchase_frequency", expr("round(transactions_count / (customer_tenure_days / 30.0), 2)"))
    
    # Generamos emails con problemas de calidad intencionales
    df = df \
        .withColumn("email_domain", expr("element_at(array('gmail.com', 'hotmail.com', 'outlook.com', 'yahoo.com', 'empresarial.co', 'corporativo.com', null), int(rand() * 7) + 1)")) \
        .withColumn("email", 
                    expr("""case 
                        when rand() > 0.05 and email_domain is not null 
                        then concat(lower(first_name), '.', lower(last_name), '@', email_domain)
                        else null end""")) \
        .drop("email_domain")
    
    # Generamos teléfonos con formatos inconsistentes
    df = df \
        .withColumn("phone_rand", expr("rand()")) \
        .withColumn("phone_number", expr("concat(string(int(rand() * 50) + 300), string(int(rand() * 9000000) + 1000000))")) \
        .withColumn("phone", 
                    expr("""case 
                        when phone_rand < 0.25 then concat('+57 ', phone_number)
                        when phone_rand < 0.50 then concat('57', phone_number)
                        when phone_rand < 0.75 then phone_number
                        else null end""")) \
        .drop("phone_rand", "phone_number")
    
    # Añadimos columnas adicionales para retos de calidad de datos
    df = df \
        .withColumn("duplicate_flag", expr("case when rand() < 0.02 then true else false end")) \
        .withColumn("data_source", expr("element_at(array('CRM', 'Web', 'Mobile App', 'Call Center', 'Third Party'), int(rand() * 5) + 1)")) \
        .withColumn("updated_at", expr("timestamp(date_add(registration_date, int(rand() * 100)))")) \
        .withColumn("loyalty_points", expr("int(total_spent * 0.01)")) \
        .withColumn("preferred_contact_method", expr("element_at(array('Email', 'Phone', 'SMS', 'WhatsApp'), int(rand() * 4) + 1)")) \
        .withColumn("marketing_consent", expr("rand() > 0.3")) \
        .withColumn("customer_segment", lit(None).cast(StringType())) \
        .withColumn("lifetime_value", lit(None).cast(IntegerType())) \
        .withColumn("churn_probability", expr("round(rand(), 2)")) \
        .drop("education_multiplier", "base_salary")
    
    # Reordenamos columnas para mejor legibilidad
    df = df.select(
        "customer_id", "first_name", "last_name", "email", "phone", "age", "gender", "marital_status",
        "dependents_count", "city", "department", "profession", "education_level", "years_employed",
        "contract_type", "company_sector", "company_size", "remote_work", "registration_date",
        "last_purchase_date", "last_login_date", "customer_tenure_days", "days_since_last_purchase",
        "value_day", "days_worked", "monthly_income", "net_monthly_income", "bonus_percentage",
        "annual_bonus", "tax_percentage", "monthly_tax", "savings_rate", "monthly_savings",
        "debt_amount", "debt_to_income_ratio", "credit_score", "risk_category", "has_vehicle",
        "has_property", "transactions_count", "avg_transaction_value", "total_spent",
        "purchase_frequency", "payment_method", "loyalty_points", "is_active", "churn_probability",
        "preferred_contact_method", "marketing_consent", "data_source", "duplicate_flag",
        "updated_at", "customer_segment", "lifetime_value"
    )
    
    return df

# Generamos 1 millón de registros (será mucho más rápido que el loop)
df_persona = generate_customer_spark(1000000)

In [0]:

# # Renombrar columnas para eliminar espacios
# df_renamed = df_persona \
#     .withColumnRenamed("Customer ID", "customer_id") \
#     .withColumnRenamed("First Name", "first_name") \
#     .withColumnRenamed("Last Name", "last_name") \
#     .withColumnRenamed("Email", "email") \
#     .withColumnRenamed("Phone", "phone") \
#     .withColumnRenamed("Age", "age") \
#     .withColumnRenamed("City", "city") \
#     .withColumnRenamed("Department", "department") \
#     .withColumnRenamed("Profession", "profession") \
#     .withColumnRenamed("Education Level", "education_level") \
#     .withColumnRenamed("Contract Type", "contract_type") \
#     .withColumnRenamed("Company Sector", "company_sector") \
#     .withColumnRenamed("Registration Date", "registration_date") \
#     .withColumnRenamed("Last Purchase Date", "last_purchase_date") \
#     .withColumnRenamed("Value Day", "value_day") \
#     .withColumnRenamed("Days Worked", "days_worked") \
#     .withColumnRenamed("Monthly Income", "monthly_income") \
#     .withColumnRenamed("Transactions Count", "transactions_count") \
#     .withColumnRenamed("Avg Transaction Value", "avg_transaction_value") \
#     .withColumnRenamed("Total Spent", "total_spent") \
#     .withColumnRenamed("Payment Method", "payment_method") \
#     .withColumnRenamed("Active", "active") \
#     .withColumnRenamed("Customer Segment", "customer_segment") \
#     .withColumnRenamed("Lifetime Value", "lifetime_value")

1. usuario con mayor y menor venta
2. Usuario con mayor y menor ingreso
3. Promedio por venta e ingreso
4. Cuantos usuarios por ciudad
5. Que ciudad ha tenido mayores compras y sus promedio
6. Que ciudad ha tenido mejor ingreso y sus promedio

## Guardamos la tabla en el catalogo

In [0]:
# Drop the table first to avoid schema conflicts
# spark.sql("DROP TABLE IF EXISTS workspace.employees.new_table_user")

# Now save the DataFrame as a new table
df_persona.write.format("delta").mode("overwrite").saveAsTable("workspace.employees.new_table_user")

In [0]:
data = spark.table("workspace.employees.new_table_user")

In [0]:
display(data)

In [0]:
# spark.sql("DROP TABLE IF EXISTS workspace.employees.table_user")